## In this notebook we create the dataset roles_df
which contains a slightly cleaned up version of role_and_frame_annoations.tsv

**Hero**: does the story suggest that some entity could alleviate the problem?
(actively alleviating, or planning to alleviate, the problem)

**Victim**: Does the story emphasize how one or more entities are NEGATIVELY affected by the issue/problem?

**Villain**: Does the story suggest that some entity is responsible for the issue/problem?
(actively causing or having caused the problem)



In [1]:
import pandas as pd


In [2]:
articles_metadata = pd.read_csv(
    '../data/external/articles_metadata.tsv', 
    sep='\t').copy()

articles_df = articles_metadata.loc[:, ["ID", "mbfc", "text"]]

In [3]:
role_annotations_df = pd.read_csv(
    '../data/external/role_and_frame_annoations.tsv', 
    sep='\t',
    encoding='ascii',
    encoding_errors='ignore',  # Skip non-ASCII characters
    ).copy()

role_annotations_df.head()

,ID,entity,group,role,mbfc,RE,HI,CO,MO,EC
0,0,climate alarmists,ENV.ORGS_ACTIVISTS,villain,right_bias,False,False,True,True,False
1,0,they,ENV.ORGS_ACTIVISTS,villain,right_bias,False,False,True,True,False
2,0,people who are unscientific,GENERALPUBLIC,victim,right_bias,False,False,True,True,False
3,2,climate alarmism,ENV.ORGS_ACTIVISTS,villain,right_bias,False,False,False,False,False
4,2,Democratic New York Rep. Alexandria Ocasio,GOVERNMENTS_POLITICIANS_POLIT.ORGS,villain,right_bias,False,False,False,False,False


In [4]:
# Remove rows where group is CLIMATECHANGE and its narrative role is victim

role_annotations_df = role_annotations_df[~((role_annotations_df['group'] == 'CLIMATECHANGE') & (role_annotations_df['role'] == 'victim'))]

# keep only columns of interest 

roles_df = role_annotations_df.loc[:, ["ID", "entity", "role"]]


In [5]:
# remove trailing whitespaces

roles_df["entity"] = roles_df.entity.str.strip()

In [6]:
# drop duplicates

roles_df = roles_df.drop_duplicates()

In [8]:
# Remove entities that aren't informative

entities_to_remove =[

    "they", "we", 
    "her", "his", "He", "His",
    "its", "We", "us", "he",
    "our", "I", "many", 

]

# Remove rows where entity is in entities_to_remove
roles_df = roles_df[~roles_df['entity'].isin(entities_to_remove)]

roles_df.shape[0]

1882

some entities have conflicting roles: entity spelling matches exactly, but it has conflicting roles (hero and victim and villain). That's possibly justifiable in some of the
cases, but sometimes it's fully contradictory, and the annotators didn't provide reasons. To be safe, we'll eliminate them from the dataset

In [9]:
# Group by ID and entity, then filter for cases where there are multiple different roles
conflicting_roles = roles_df.groupby(['ID', 'entity']).filter(lambda x: x['role'].nunique() > 1)

# Sort by ID and entity for better readability
conflicting_roles = conflicting_roles.sort_values(['ID', 'entity'])

conflicting_roles

,ID,entity,role
16,9,Republican Rep. Matt Gaetz,hero
17,9,Republican Rep. Matt Gaetz,villain
40,20,the ocean,hero
41,20,the ocean,villain
50,25,the world's,hero
...,...,...,...
1873,850,Bloomberg,villain
1870,850,Trump,hero
1871,850,Trump,villain
1917,875,President Trump,hero


In [10]:
roles_df = roles_df[~roles_df.index.isin(conflicting_roles.index)]

Now we remove by hand some examples that don't make sense, which we got by going through each entry

In [11]:
odd_entities = [
    "3,600 premature deaths a year, 1,700 heart attacks and 90,000 asthma attacks",
    "could also be radical environmentalists more broadly-however they are not mentioned as often",
    "that it's melting faster than they thought .",
    "face temperatures of 90F 32C and higher",
    "for subsidizing the National Flood Insurance Program",
    "The team's",
]

In [12]:
roles_df = roles_df[~roles_df.entity.isin(odd_entities)]

roles_df.shape[0]

1705

### Drop specific cases

In [13]:
roles_df[(roles_df.entity == "kids") & (roles_df.role == "villain")].index

Index([81], dtype='int64')

In [14]:
roles_df.drop(81, inplace=True)

In [15]:
roles_df[roles_df.entity == "ruled"]

,ID,entity,role
1516,674,ruled,villain


In [16]:
roles_df.drop(1516, inplace=True)

Some remediation for very long entities

In [17]:
# suspicious: very long entities

roles_df[roles_df['entity'].str.split().str.len() > 12]


,ID,entity,role
246,109,3 million species of plants and animals and 1 ...,victim
673,278,Elizabeth Warren Promises Day One Executive Or...,hero
900,375,Trump Says 'Bad Environmental Laws' Made Deadl...,hero
1009,418,President Trump Says He's Too Intelligent To B...,villain


In [18]:
# obvious remediation

roles_df.at[673, 'entity'] = "Elizabeth Warren"

roles_df.at[1009, 'entity'] = "President Trump"

roles_df.at[900, 'entity'] = "Trump"


This is a more subtle deduplication effort: 

Same entity and same role, just different case or articles. Keep the shortest. Example:
Amazon rainforest, victim
The Amazon Rainforest, victim

In [19]:
roles_df[roles_df.ID == 161]

,ID,entity,role
377,161,Amazon rainforest,victim
378,161,The Amazon Rainforest,victim
379,161,President Trump,villain
380,161,The G-7,hero


In [20]:
roles_df.drop(378, inplace=True)

In [22]:
roles_df[roles_df.ID == 161]

,ID,entity,role
377,161,Amazon rainforest,victim
379,161,President Trump,villain
380,161,The G-7,hero


In [24]:
roles_df[roles_df.ID == 115]

,ID,entity,role
272,115,federal and Native American land,victim
273,115,public health,victim
274,115,The Trump administration,villain
275,115,Trump administration,villain
276,115,greenhouse gas emissions,villain
277,115,Clean Power Plan,hero


In [25]:
roles_df.drop(274, inplace=True)

In [26]:
roles_df[roles_df.ID == 278]

,ID,entity,role
669,278,Elizabeth Warren,hero
670,278,fossil fuels,villain
671,278,Fracking,villain
673,278,Elizabeth Warren,hero


In [27]:
roles_df.drop(673, inplace=True)

In [28]:

roles_df[roles_df.ID == 77]

,ID,entity,role
152,77,Perpetual presidential candidate Bernie Sanders,villain
153,77,Farmers,victim
154,77,farmers,victim


In [29]:
roles_df.drop(153, inplace=True)

In [30]:
roles_df[roles_df.ID == 93]

,ID,entity,role
192,93,Rural voters,victim
193,93,The people,victim
194,93,Trump,villain
195,93,Farmers,victim
196,93,farmers,victim


In [31]:
roles_df.drop(195, inplace=True)

In [33]:
roles_df[roles_df.ID == 184]

,ID,entity,role
441,184,The Trump administration,villain
442,184,Trump administration,villain
443,184,Callifornia,villain
444,184,Car manufacturers,victim


In [34]:
roles_df.drop(441,inplace=True)

In [35]:
roles_df[roles_df.ID == 210]

,ID,entity,role
497,210,Climate asylum seekers,victim
501,210,his home state of Texas,victim
502,210,carbon emissions,villain
503,210,Carbon emissions,villain
504,210,Cap-and-trade system,hero


In [36]:
roles_df.drop(503, inplace=True)

In [37]:
roles_df[roles_df.ID == 253]

,ID,entity,role
621,253,Amazon rainforest,victim
622,253,The Amazon Rainforest,victim
623,253,Wildfires,villain
624,253,G7 nations,hero
625,253,Macron,hero
626,253,President Bolsonaro,villain


In [38]:
roles_df.drop(622, inplace=True)

In [39]:
roles_df[roles_df.ID == 425]

,ID,entity,role
1019,425,Small islands,victim
1020,425,small islands,victim
1021,425,major coastal metropolises,victim
1022,425,the U.S.,villain
1023,425,Republicans,villain
1024,425,U.S climate response,victim
1025,425,the Intergovernmental Panel on Climate Change,hero


In [40]:
roles_df.drop(1019, inplace=True)

In [41]:
roles_df[roles_df.ID == 468]

,ID,entity,role
1119,468,President Donald Trump,villain
1120,468,the Environmental Protection Agency,victim
1121,468,a significantly less notorious successor,villain
1122,468,environmental regulations,victim
1123,468,Environmental regulations,victim


In [42]:
roles_df.drop(1123, inplace=True)

In [43]:
roles_df[roles_df.ID == 532]

,ID,entity,role
1271,532,forest management,hero
1272,532,Climate change,villain
1273,532,climate change,villain
1274,532,environmental groups,villain
1275,532,environmental terrorist groups,villain
1276,532,California,victim


In [44]:
roles_df.drop(1272, inplace=True)

In [45]:
roles_df[roles_df.ID == 611]

,ID,entity,role
1386,611,environment,victim
1387,611,workers and their families,victim
1388,611,liberals,villain
1389,611,Liberals,villain
1390,611,Fox News host Tucker Carlson,hero


In [46]:
roles_df.drop(1388, inplace=True)

In [47]:
roles_df[roles_df.ID == 619]

,ID,entity,role
1400,619,people impacted by natural disasters,victim
1401,619,People impacted by natural disasters,victim
1402,619,a new study,hero


In [48]:
roles_df.drop(1401, inplace=True)

In [49]:
roles_df[roles_df.ID == 812]

,ID,entity,role
1789,812,weather phenomenon,villain
1791,812,Human activities,villain
1792,812,human activities,villain
1793,812,the Paris agreement,hero


In [50]:
roles_df.drop(1791, inplace=True)

In [51]:
roles_df[roles_df.ID == 822]

,ID,entity,role
1804,822,greenies,villain
1805,822,Climate Industrial Complex,villain
1806,822,The Climate Industrial Complex,villain
1807,822,Scientific papers casting doubt on global warming,hero


In [52]:
roles_df.drop(1806, inplace=True)

In [53]:
roles_df[roles_df.ID == 823]

,ID,entity,role
1808,823,Obama,villain
1809,823,Trump,hero
1810,823,U.S economy,victim
1811,823,US Economy,victim
1812,823,the Clean Power Plan,villain


In [54]:
roles_df.drop(1811, inplace=True)

In [55]:
roles_df[roles_df.ID == 294]

,ID,entity,role
714,294,minority communities,victim
715,294,minority communties,victim
716,294,Sen. Elizabeth Warren D-MA,hero
717,294,Industrial pollution,villain


In [56]:
roles_df.drop(714, inplace=True)

### Handle entities with commas

Because we will need a list of strings per ID and role, and comma separated values will be different entities

In [57]:

roles_df[roles_df.entity.str.contains(",")]

,ID,entity,role
34,13,"Greta Thunberg, a 16-year-old climate change a...",hero
218,97,"observational data, time series, and quasi-exp...",hero
435,180,"labor unions, oil companies",victim
468,190,"his French counterpart, Emmanuel Macron",hero
553,228,"Owner Peter Melnik, a fourth-generation dairy ...",hero
818,334,"the nearly 350,000-acre Mendocino Complex Fire",victim
914,381,"University of Colorado professor Roger Pielke,...",hero
921,383,"The ballot measure, called Initiative 1631",villain
951,399,"the US economy, environment, and human health ...",victim
956,403,"Nick Mullins, a ninth-generation Appalachian f...",victim


Manually change comma separated numbers to periods

In [58]:
roles_df.at[1637, 'entity'] = "25.000 people"

roles_df.at[1270, "entity"] = "an estimated 573.000 birds and 888.000 bats a year"

roles_df.at[687, "entity"] = "the nearly 350.000-acre Mendocino Complex Fire"

Index 378 is actually two different entities (I read the article)

In [59]:
roles_df = pd.concat([roles_df, pd.DataFrame({
    'ID': [180, 180],
    'entity': ['labor unions','oil companies'],  
    'role': ['victim', 'victim']
})]
, ignore_index=True
)

# roles_df[roles_df.ID == 180]

roles_df.drop(378, inplace=True)

Obvious replacement

In [60]:

roles_df.at[410, 'entity'] = "Emmanuel Macron"

Strip commas from the rest

In [61]:
roles_df[roles_df.entity.str.contains(",")]

,ID,entity,role
30,13,"Greta Thunberg, a 16-year-old climate change a...",hero
186,97,"observational data, time series, and quasi-exp...",hero
479,228,"Owner Peter Melnik, a fourth-generation dairy ...",hero
686,334,"the nearly 350,000-acre Mendocino Complex Fire",victim
775,381,"University of Colorado professor Roger Pielke,...",hero
780,383,"The ballot measure, called Initiative 1631",villain
809,399,"the US economy, environment, and human health ...",victim
814,403,"Nick Mullins, a ninth-generation Appalachian f...",victim
843,410,"Tyndall Air Force Base near Panama City, Florida,",victim
952,459,"the new US ambassador to Canada, Kelly Craft",villain


In [62]:
roles_df.loc[roles_df.entity.str.contains(","), 'entity'] = roles_df.loc[roles_df.entity.str.contains(","), 'entity'].str.replace(",", "")

In [63]:
roles_df.loc[roles_df.entity.str.contains(",")]

,ID,entity,role


In [67]:
roles_df[roles_df.entity.str.contains("Eskridge")]

,ID,entity,role
1158,583,"James ""Ooker"" Eskridge",hero
1215,618,James Eskridge mayor of the mainly Republican ...,hero


In [69]:
roles_df.loc[1158].entity

'James "Ooker" Eskridge'

In [70]:
roles_df.at[1158, "entity"] = 'James Ooker Eskridge'

In [ ]:
roles_df.to_csv('../data/interim/roles_df.tsv', sep='\t', index=False)